# Data Passport — Проверка данных

Паспорт очищенных датасетов. Запускается после ноутбуков 01–04.  
Цель: убедиться, что все источники загружаются корректно и связи между ними рабочие.

In [2]:
import pandas as pd
import numpy as np
import os

import help_130625_dam as h

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

CLEANED_DIR = os.path.join('..', 'data', 'cleaned')

## Загрузка очищенных данных

In [3]:
# Загружаем все .pkl файлы
contacts = pd.read_pickle(os.path.join(CLEANED_DIR, 'contacts_clean.pkl'))
deals = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))
calls = pd.read_pickle(os.path.join(CLEANED_DIR, 'calls_clean.pkl'))
spend = pd.read_pickle(os.path.join(CLEANED_DIR, 'spend_clean.pkl'))

print(f"Contacts: {contacts.shape}")
print(f"Deals: {deals.shape}")
print(f"Calls: {calls.shape}")
print(f"Spend: {spend.shape}")

Contacts: (18510, 7)
Deals: (19815, 28)
Calls: (92599, 9)
Spend: (19862, 8)


## Сводка по датасетам

In [4]:
datasets = {
    'contacts': contacts,
    'deals':    deals,
    'calls':    calls,
    'spend':    spend,
}

for name, df in datasets.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    null_str = ', '.join(f'{c}: {n}' for c, n in nulls.items()) if len(nulls) else 'нет'
    print(f"\n{'─'*55}")
    print(f"  {name.upper()}: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
    print(f"  Пропуски: {null_str}")
    print(f"  Период: {df.select_dtypes('datetime').apply(lambda s: f'{s.min().date()} → {s.max().date()}').to_dict()}")



───────────────────────────────────────────────────────
  CONTACTS: 18,510 строк × 7 столбцов
  Пропуски: first_payment_date: 17696, new_registration_date: 18487
  Период: {'created_time': '2023-06-27 → 2024-06-21', 'modified_time': '2023-07-06 → 2024-06-21', 'first_payment_date': '2023-07-04 → 2024-06-15', 'new_registration_date': '2023-07-03 → 2024-04-29'}

───────────────────────────────────────────────────────
  DEALS: 19,815 строк × 28 столбцов
  Пропуски: closing_date: 6680, sla: 6578, contact_id: 47, duration_raw: 6680, deal_duration_days: 6680
  Период: {'closing_date': '2022-10-11 → 2024-12-11', 'created_time': '2023-07-03 → 2024-06-21'}

───────────────────────────────────────────────────────
  CALLS: 92,599 строк × 9 столбцов
  Пропуски: contactid: 3799
  Период: {'call_start_time': '2023-06-30 → 2024-06-21', 'call_end_time': '2023-06-30 → 2024-06-21'}

───────────────────────────────────────────────────────
  SPEND: 19,862 строк × 8 столбцов
  Пропуски: нет
  Период: {'dat

## Проверка связей (Cross-check)

In [5]:
contact_ids = set(contacts['id'])

# Deals → Contacts (считаем только честные пропуски)
is_missing_deal = deals['contact_id'].isna()
valid_deals = deals[~is_missing_deal]
deals_matched = valid_deals['contact_id'].isin(contact_ids).sum()

print(f"Deals → Contacts:  {deals_matched:,} | {len(valid_deals):,} ({deals_matched/len(valid_deals)*100:.1f}%)  "
      f"[+ {is_missing_deal.sum():,} сделок без contact_id (NaN)]")

# Calls → Contacts
is_missing_call = calls['contactid'].isna()
valid_calls = calls[~is_missing_call]
calls_matched = valid_calls['contactid'].isin(contact_ids).sum()

print(f"Calls → Contacts:  {calls_matched:,} | {len(valid_calls):,} ({calls_matched/len(valid_calls)*100:.1f}%)  "
      f"[+ {is_missing_call.sum():,} звонков без contactid (NaN)]")


Deals → Contacts:  19,768 | 19,768 (100.0%)  [+ 47 сделок без contact_id (NaN)]
Calls → Contacts:  88,800 | 88,800 (100.0%)  [+ 3,799 звонков без contactid (NaN)]


In [6]:
# Анализ пересечения по источнику и кампании (Spend vs Deals)
spend_keys = spend[['source', 'campaign']].drop_duplicates()
deals_keys = deals[['source', 'campaign']].drop_duplicates()

print(f"Уникальных пар (Source + Campaign) в Spend: {len(spend_keys)}")
print(f"Уникальных пар (Source + Campaign) в Deals: {len(deals_keys)}")

# Пересечение
merged_keys = pd.merge(spend_keys, deals_keys, on=['source', 'campaign'], how='inner')
print(f"Совпадающих пар: {len(merged_keys)}")

print("\nТоп-10 совпадающих источников:")
display(merged_keys['source'].value_counts().head(10))

# Что есть в расходах, но нет в сделках
missing_in_deals = pd.merge(spend_keys, deals_keys, on=['source', 'campaign'], how='left', indicator=True)
missing_in_deals = missing_in_deals[missing_in_deals['_merge'] == 'left_only']

print(f"\nПар в Spend, которых нет в Deals: {len(missing_in_deals)}")
display(missing_in_deals.head(5))

Уникальных пар (Source + Campaign) в Spend: 66
Уникальных пар (Source + Campaign) в Deals: 355
Совпадающих пар: 55

Топ-10 совпадающих источников:


source
Facebook Ads    22
Google Ads       8
Tiktok Ads       7
Webinar          6
Youtube Ads      4
Test             2
CRM              1
Bloggers         1
SMM              1
Organic          1
Name: count, dtype: int64


Пар в Spend, которых нет в Deals: 11


,source,campaign,_merge
3,Google Ads,Unknown,left_only
30,Offline,Unknown,left_only
34,Webinar,15.11.23wide_webinar_DE,left_only
35,Webinar,blog2_DE,left_only
36,Facebook Ads,30.11.23wide_DE,left_only


In [7]:
# Сравнение: Объединение ТОЛЬКО по источнику (Source)
spend_sources = spend['source'].unique()
deals_sources = deals['source'].unique()

print(f"Уникальных источников в Spend: {len(spend_sources)}")
print(f"Уникальных источников в Deals: {len(deals_sources)}")

# Пересечение только по источнику
source_match = set(spend_sources).intersection(set(deals_sources))
print(f"Совпадающих источников: {len(source_match)}")
print(f"Список совпадающих: {sorted(list(source_match))}")

# Анализ охвата затрат (какой % денег мы сможем распределить по сделкам)
total_spend = spend['spend'].sum()
matched_spend = spend[spend['source'].isin(deals_sources)]['spend'].sum()
print(f"\n% затрат, которые находят свой источник в сделках: {matched_spend/total_spend*100:.1f}%")

# Анализ охвата сделок (какой % сделок найдет свои затраты)
total_deals = len(deals)
matched_deals = deals[deals['source'].isin(spend_sources)].shape[0]
print(f"% сделок, которые находят свои затраты по источнику: {matched_deals/total_deals*100:.1f}%")

# Что потеряется (в Spend есть, в Deals нет)
missing_source = set(spend_sources) - set(deals_sources)
if missing_source:
    print(f"\nИсточники в Spend, которых НЕТ в Deals: {missing_source}")
    display(spend[spend['source'].isin(missing_source)].groupby('source')['spend'].sum().to_frame())

Уникальных источников в Spend: 14
Уникальных источников в Deals: 14
Совпадающих источников: 13
Список совпадающих: ['Bloggers', 'CRM', 'Facebook Ads', 'Google Ads', 'Offline', 'Organic', 'Partnership', 'SMM', 'Telegram posts', 'Test', 'Tiktok Ads', 'Webinar', 'Youtube Ads']

% затрат, которые находят свой источник в сделках: 99.8%
% сделок, которые находят свои затраты по источнику: 99.8%

Источники в Spend, которых НЕТ в Deals: {'Radio'}


,spend
source,
Radio,300.00


In [8]:
# Сравнение: Объединение ТОЛЬКО по кампаниям (Campaign)
spend_campaigns = spend['campaign'].unique()
deals_campaigns = deals['campaign'].unique()

print(f"Уникальных кампаний в Spend: {len(spend_campaigns)}")
print(f"Уникальных кампаний в Deals: {len(deals_campaigns)}")

# Пересечение только по кампаниям
camp_match = set(spend_campaigns).intersection(set(deals_campaigns))
print(f"Совпадающих названий кампаний: {len(camp_match)}")

# Анализ охвата затрат по кампаниям
matched_spend_camp = spend[spend['campaign'].isin(deals_campaigns)]['spend'].sum()
print(f"\n% затрат, которые находят свою кампанию в сделках: {matched_spend_camp/total_spend*100:.1f}%")

# Анализ охвата сделок по кампаниям
matched_deals_camp = deals[deals['campaign'].isin(spend_campaigns)].shape[0]
print(f"% сделок, которые находят свои затраты по кампании: {matched_deals_camp/total_deals*100:.1f}%")

# Проверка на риск: есть ли одинаковые названия кампаний у разных источников?
camp_source_check = spend.groupby('campaign')['source'].nunique()
multi_source_camps = camp_source_check[camp_source_check > 1]
if not multi_source_camps.empty:
    print(f"\n[!] Найдено кампаний с одинаковым именем у разных источников: {len(multi_source_camps)}")
    display(multi_source_camps.to_frame())
else:
    print("\n[OK] Все названия кампаний уникальны в рамках своих источников.")

Уникальных кампаний в Spend: 52
Уникальных кампаний в Deals: 152
Совпадающих названий кампаний: 46

% затрат, которые находят свою кампанию в сделках: 98.6%
% сделок, которые находят свои затраты по кампании: 68.5%

[!] Найдено кампаний с одинаковым именем у разных источников: 2


,source
campaign,
07.12.23test_DE,2
Unknown,14


In [9]:
# ПРАВИЛЬНОЕ ОБЪЕДИНЕНИЕ: Агрегируем раздельно, затем джойним агрегаты
#
# ПОЧЕМУ НЕ ДЕЛАЕМ deals LEFT JOIN spend_agg по [date, source, campaign]:
# Если за один день из одной кампании пришли N сделок, а расходы = 100€,
# то каждая сделка получит свои 100€, и sum(spend) даст N×100€ — дублирование.
# Пример выше: 366,329 € вместо реальных 149,523 € (x2.5 ошибка).
#
# ПРАВИЛЬНЫЙ ПУТЬ: агрегировать deals и spend независимо, потом джойнить агрегаты.

total_spend_actual = spend['spend'].sum()
print(f"Фактические расходы в Spend: {total_spend_actual:,.2f} €")

# 1. Агрегируем сделки по [Source + Campaign]
deals_agg = (
    deals.groupby(['source', 'campaign'], observed=True)
    .agg(
        deals_count    = ('id',                  'count'),
        buyers_count   = ('is_buyer',            'sum'),   # T — транзакции покупателей
        revenue        = ('initial_amount_paid', 'sum'),   # Rev — выручка от покупателей
    )
    .reset_index()
)

# 2. Агрегируем расходы по [Source + Campaign]
spend_agg = (
    spend.groupby(['source', 'campaign'], observed=True)
    .agg(
        total_spend   = ('spend',       'sum'),
        total_clicks  = ('clicks',      'sum'),
        total_impr    = ('impressions', 'sum'),
    )
    .reset_index()
)

# 3. Объединяем агрегаты (1:1 джойн — никакого дублирования)
campaign_romi = pd.merge(deals_agg, spend_agg, on=['source', 'campaign'], how='outer').fillna(0)

# Считаем метрики
campaign_romi['conversion_rate'] = (campaign_romi['buyers_count'] / campaign_romi['deals_count'].replace(0, np.nan) * 100).round(1).fillna(0)
campaign_romi['romi']            = ((campaign_romi['revenue'] - campaign_romi['total_spend']) / campaign_romi['total_spend'].replace(0, np.nan) * 100).round(1).fillna(0)
campaign_romi['cac']             = (campaign_romi['total_spend'] / campaign_romi['buyers_count'].replace(0, np.nan)).round(0).fillna(0)
campaign_romi['avg_check']       = (campaign_romi['revenue'] / campaign_romi['buyers_count'].replace(0, np.nan)).round(0).fillna(0)  # AOV = Rev / T
campaign_romi = campaign_romi.sort_values('revenue', ascending=False)

# Проверяем: сумма расходов должна совпасть с оригинальным Spend
spend_check = campaign_romi['total_spend'].sum()
print(f"Расходы после агрегации (должно совпасть): {spend_check:,.2f} € {'[OK]' if abs(spend_check - total_spend_actual) < 1 else '[!] РАСХОЖДЕНИЕ'}")

print(f"\nИтоговых пар Source+Campaign: {len(campaign_romi)}")
print(f"Из них платных (Spend > 0): {(campaign_romi['total_spend'] > 0).sum()}")
print(f"Из них принесших выручку (Revenue > 0): {(campaign_romi['revenue'] > 0).sum()}")

print("\n--- ТОП-10 КАМПАНИЙ ПО ВЫРУЧКЕ ---")
display(campaign_romi[['source', 'campaign', 'deals_count', 'buyers_count', 'conversion_rate',
                        'revenue', 'avg_check', 'total_spend', 'romi', 'cac']].head(10))

Фактические расходы в Spend: 149,523.45 €
Расходы после агрегации (должно совпасть): 149,523.45 € [OK]

Итоговых пар Source+Campaign: 366
Из них платных (Spend > 0): 58
Из них принесших выручку (Revenue > 0): 175

--- ТОП-10 КАМПАНИЙ ПО ВЫРУЧКЕ ---


,source,campaign,deals_count,buyers_count,conversion_rate,revenue,avg_check,total_spend,romi,cac
170,Organic,Unknown,1313,121.00,9.20,466410.00,3855.00,0.00,0.00,0.00
143,Google Ads,performancemax_digitalmarkt_ru_DE,2571,107.00,4.20,412250.00,3853.00,0.00,0.00,0.00
230,SMM,Unknown,1144,69.00,6.00,304950.00,4420.00,7269.52,4094.90,105.00
315,Tiktok Ads,12.07.2023wide_DE,1546,48.00,3.10,276000.00,5750.00,9471.52,2814.00,197.00
364,Youtube Ads,youtube_shorts_DE,1592,52.00,3.30,246700.00,4744.00,14149.22,1643.60,272.00
101,Facebook Ads,02.07.23wide_DE,948,51.00,5.40,192450.00,3774.00,6913.60,2683.60,136.00
104,Facebook Ads,04.07.23recentlymoved_DE,735,30.00,4.10,138600.00,4620.00,4523.31,2964.10,151.00
113,Facebook Ads,12.09.23interests_Uxui_DE,516,25.00,4.80,136950.00,5478.00,3753.06,3549.00,150.00
106,Facebook Ads,07.07.23LAL_DE,531,28.00,5.30,124800.00,4457.00,4200.37,2871.20,150.00
136,Google Ads,Dis_DE,567,29.00,5.10,117950.00,4067.00,0.00,0.00,0.00


In [10]:
# Сохраняем агрегат как единый источник правды для 06 и 07
CAMPAIGN_ROMI_OUTPUT = os.path.join(CLEANED_DIR, 'campaign_romi.pkl')

os.makedirs(CLEANED_DIR, exist_ok=True)
campaign_romi.to_pickle(CAMPAIGN_ROMI_OUTPUT)
campaign_romi.to_excel(CAMPAIGN_ROMI_OUTPUT.replace('.pkl', '.xlsx'), index=False)

print(f"Сохранено: {CAMPAIGN_ROMI_OUTPUT}")
print(f"Строк: {len(campaign_romi)}, Колонок: {len(campaign_romi.columns)}")
# print(f"Колонки: {list(campaign_romi.columns)}")

Сохранено: ../data/cleaned/campaign_romi.pkl
Строк: 366, Колонок: 12


## Выводы

При попытке объединить данные сделок и расходов через прямой `deals LEFT JOIN spend` по `[date, source, campaign]` был обнаружен **критический баг дублирования расходов**: если за один день из одной кампании пришло N сделок, каждая из них получала 100% суммы расходов — итоговая сумма оказывалась в N раз больше реальной. В данных это дало **366 329 € вместо фактических 149 523 €** (×2.5 завышение). Причина — M:1 соотношение при джойне: один агрегат расходов матчится на несколько строк сделок.

Без исправления ROMI по всем каналам был бы занижен в разы (расходы казались бы непропорционально большими относительно выручки), CAC — завышен, а любые управленческие решения по бюджету каналов принимались бы на ложных данных.

**Что сделано:** применён правильный паттерн — сделки и расходы агрегируются **независимо** по `[source, campaign]`, затем агрегаты соединяются 1:1. Результат сохранён в `campaign_romi.pkl` (366 пар источник+кампания, 12 метрик). Сверка со Spend подтвердила корректность: расходы совпали до копейки `[OK]`.

Дополнительно установлено:
- Объединение по одному только `source` даёт **99.8% покрытия** сделок и расходов — достаточно для агрегированного анализа каналов.
- Объединение по `campaign` без `source` рискованно: одно и то же название кампании встречается в нескольких источниках, что приводит к неверной атрибуции.
- Оптимальный ключ объединения — **`[source, campaign]`** как компромисс точности и покрытия.

> **Архитектурное решение:** `campaign_romi.pkl` является **единственным источником правды** для расчёта ROMI в ноутбуках 06 и 07 — они не пересчитывают объединение заново, а загружают готовый агрегат.